In [39]:

from typing import List
from config import CellConfig
'''基因转录
A T C G

AA = d; AT = a; AC = w; AG = s
TA = +; TT = -; TC = ,; TG = .
CA = [; CT = ]; CC = >; CG = <
GA = start/pause; GT = stop; GC = rep

!d = a;  ?d = w ;  !?d = ?!d = s
!+ = -;  ?+ = , ;  !?+ = ?!+ = .
![ = ];  ?[ = > ;  !?[ = ?![ = <
!!/?? ： pause
'''

def _cut_DNAs(DNA : str) -> List[str]:
    '''将基因序列切割为片段'''
    s = DNA
    tokens = []
    base_set = ['d', '+', '[']
    stop_set = ['??']
    duo_set = ['!d', '?d', '!+','?+','![', '?[', '!!', '??']
    tri_set = ['!?d', '?!d', '!?+', '?!+', '!?[', '?![']
    fraction = []
    i = 0
    while i < len(DNA):
        if i + 2 <= len(s) and s[i:i+2] in stop_set:
            break
        # 三字符
        if i + 3 <= len(s) and s[i:i+3] in tri_set:
            tokens.append(s[i:i+3])
            i += 3
            continue
        # 两字符
        if i + 2 <= len(s) and s[i:i+2] in duo_set:
            tokens.append(s[i:i+2])
            i += 2
            continue
        # 单字符
        if s[i] in base_set:
            tokens.append(s[i])
            i += 1
            continue
        # 噪声跳过
        i += 1

    return tokens
        
def _translate_cutted_DNA(cutted_DNA: List[str]) -> List[str]:
    '''将切割后的DNA转录为RNA'''
    translate_dict = {
        'd' : 'd', '!d' : 'a', '?d' : 'w', '!?d' : 's', '?!d' : 's',
        '+' : '+', '!+' : '-', '?+' : ',', '!?+' : '.', '?!+' : '.',
        '[' : '[', '![' : ']', '?[' : '>', '!?[' : '<', '?![' : '<',
        '!!' : 'p'
    }
    translated_RNA = [translate_dict[token] for token in cutted_DNA if token in translate_dict]
    return translated_RNA

def _match_RNA(translated_RNA: List[str]) -> List[str]:
    '''为括号添加跳转位置，丢弃没有配对的括号'''
    stack = []
    matched_indices = set()  # 记录成功配对的索引
    
    # 第一遍：找出所有配对的括号
    for i, command in enumerate(translated_RNA):
        if command == '[':
            stack.append(i)
        elif command == ']':
            if len(stack) > 0:
                j = stack.pop()
                matched_indices.add(i)
                matched_indices.add(j)
    
    # 第二遍：构建结果，只保留配对的括号和非括号命令
    result = []
    index_map = {}  # 旧索引到新索引的映射
    
    for i, command in enumerate(translated_RNA):
        if command in ['[', ']']:
            if i in matched_indices:
                index_map[i] = len(result)
                result.append(command)
        else:
            result.append(command)
    
    # 第三遍：为配对的括号添加跳转位置
    stack = []
    for i, command in enumerate(result):
        if command == '[':
            stack.append(i)
        elif command == ']':
            if len(stack) > 0:
                j = stack.pop()
                result[j] = f'[{i}'
                result[i] = f']{j}'
    
    return result

def process(DNA: str) -> List[str]:
    RNA = _cut_DNAs(DNA)
    print(f"cutted_DNA: {DNA}")
    RNA = _translate_cutted_DNA(RNA)
    print(f"translated_RNA: {RNA}")
    RNA = _match_RNA(RNA)
    print(f"matched_RNA: {RNA}")

    return RNA
    
    


In [35]:
def iprocess(RNA: list[str]) -> str:
    # 将RNA字符串翻译回DNA字符串
    translate_dict = {
    'd' : 'd', '!d' : 'a', '?d' : 'w', '!?d' : 's', '?!d' : 's',
    '+' : '+', '!+' : '-', '?+' : ',', '!?+' : '.', '?!+' : '.',
    '[' : '[', '![' : ']', '?[' : '>', '!?[' : '<', '?![' : '<',
    '!!' : 'p'}
    reverse_translate_dict = {v: k for k, v in translate_dict.items()}
    DNA_parts = []
    for command in RNA:
        if command in reverse_translate_dict.keys():
            DNA_parts.append(reverse_translate_dict[command])
    return ''.join(DNA_parts)

In [40]:
DNA = '?[+[?+?![?![[!+?[?[!['
RNA = process(DNA)
print(RNA)

cutted_DNA: ?[+[?+?![?![[!+?[?[![
translated_RNA: ['>', '+', '[', ',', '<', '<', '[', '-', '>', '>', ']']
matched_RNA: ['>', '+', ',', '<', '<', '[9', '-', '>', '>', ']5']
['>', '+', ',', '<', '<', '[9', '-', '>', '>', ']5']


In [41]:
RNA = ['>', '+', '[', ',', '<', '<', '-', '>', '>', ']']
DNA = iprocess(RNA)
print(DNA)

?[+[?+?![?![!+?[?[![
